# CFP Ranker '26: Data Collection
This notebook maintains one SQLite database, `espn_cfb_2026.db`, as the **source of truth** for raw ESPN data. Running it again only calls ESPN for data that is missing or may have changed. Team defensive stats are also stored here, derived two ways: by summing player box scores, and by taking each opponent's team box score. Both they are recomputed from the database on every run, with no ESPN requests. Statistics built on top of this data (team summaries, defensive rankings, validation) live in `cfb2026_stats.ipynb`; run this notebook first to bring the database up to date.

## When each kind of data is fetched

| Data | Fetched when |
| --- | --- |
| FBS teams | The `Teams` table has no FBS teams yet (or `force=True`) |
| Daily scoreboard | The day was never fetched; or it is within the last 7 / next 14 days and was last fetched over 24 hours ago; or it was fetched before that day's games could have finished |
| Game summary (box score + play-by-play) | A Final game has no stored box score or no stored play-by-play; or ESPN had not published one of them yet and the last check was over 24 hours ago; or the game kicked off within the last 7 days and the last check was over 24 hours ago |
| Injury reports (kickoffstatus.com) | A game page listed in the site's sitemap was never fetched; or the game is within the last 7 days or upcoming and the sitemap says the page changed since our copy |

Both come from the same ESPN summary request, so one request per game fills both. A re-checked box score or play-by-play is only rewritten if ESPN's data actually changed (compared by hash).

## Tables

| Table | Purpose |
| --- | --- |
| `Teams` | FBS teams plus any FCS opponents seen in games |
| `Games` | Every scoreboard event (any status) |
| `Box_Scores` | One row per game, team and statistic, parsed from the raw payloads |
| `Raw_Boxscores` | The raw ESPN `boxscore` JSON per game (compressed), its hash and when it was fetched. `Box_Scores` can be rebuilt from this with no network calls |
| `Scoreboard_Fetches` | Which days of the scoreboard have been fetched, and when |
| `Player_Box_Scores` | One row per game, team, player, category and statistic, parsed from the raw payloads |
| `Player_Boxscore_Fetches` | The raw payload hash each game's player rows were last parsed from |
| `Raw_Plays` | The raw ESPN `drives` and `scoringPlays` JSON per game (compressed), its hash and when it was fetched. `Plays` and `Scoring_Plays` can be rebuilt from this with no network calls |
| `Plays` | One row per play (columns from `map_schema.flatten_plays`), parsed from `Raw_Plays` |
| `Scoring_Plays` | One row per score: who scored, how many points, and whether the scoring team had the ball when the play started (`is_offense_score`) |
| `Raw_Injury_Pages` | The raw kickoffstatus game page (compressed), the ESPN `game_id` it was matched to, and when it was fetched. `Injury_Reports` can be rebuilt from this with no network calls |
| `Injury_Reports` | One row per game, team and player on that game's availability report: the official conference report (`official = 1`) or an unofficial media note (`official = 0`) |
| `Team_Defensive_Stats` | Per game and team, the sum of each `defensive` / `interceptions` player stat. Rebuilt from `Player_Box_Scores` on every run |
| `Team_Defense_Opponent_Stats` | Per game and team, the opponent's `Box_Scores` stats renamed from the defense's point of view (e.g. `totalYardsAllowed`, `takeaways`) plus `pointsAllowed`, and `offensePointsAllowed` from `Scoring_Plays`. Rebuilt from `Box_Scores`, `Games` and `Scoring_Plays` on every run |

**First run on an existing database:** raw payloads were not stored before, so every Final game is fetched once (about 700 requests) to fill `Raw_Boxscores`. After that, runs make few or no requests. The same applies to play-by-play: the first run after it was added fetches every Final game once (about 700 requests) to fill `Raw_Plays`.

**Injury reports, first run:** every game page kickoffstatus has (about 600 since Week 0) is fetched once, at 0.5 s per request, so the first run takes about 5 minutes. Later runs only refetch this week's games.

## 1. Setup

In [1]:
import hashlib
import json
import sqlite3
import time
import zlib
from collections import Counter
from datetime import date, datetime, timedelta, timezone
from datetime import time as dtime
from IPython.display import display, Markdown
 
import pandas as pd
import requests

# Play-by-play parsing lives in map_schema.py so the notebook and the schema script share one definition.
from map_schema import collect_drives, flatten_plays
 
# --- Season and database ---
YEAR = 2026
DB_FILE = f"espn_cfb_{YEAR}.db"
SEASON_START = date(YEAR, 8, 1)
SEASON_END = date(YEAR + 1, 2, 1)
 
# --- Refresh rules ---
RECENT_DAYS = 7            # games/days this recent are re-checked for revisions
LOOKAHEAD_DAYS = 14        # upcoming days whose schedules are kept fresh
REFRESH_AFTER_HOURS = 24   # minimum time between re-checks of the same item
REQUEST_DELAY = 0.15       # seconds to wait after each API call
 
# --- ESPN endpoints ---
SCOREBOARD_URL = "https://site.web.api.espn.com/apis/site/v2/sports/football/college-football/scoreboard"
SUMMARY_URL = "https://site.api.espn.com/apis/site/v2/sports/football/college-football/summary"
STANDINGS_URL = f"https://site.web.api.espn.com/apis/v2/sports/football/college-football/standings?season={YEAR}"
HEADERS = {
    "User-Agent": "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/122.0.0.0 Safari/537.36",
    "Accept": "application/json",
}

_session = requests.Session()
_session.headers.update(HEADERS)
api_calls = 0  # every network request goes through api_get, so this counts them all

def utcnow():
    return datetime.now(timezone.utc)
 
def iso(dt):
    return dt.isoformat(timespec="seconds")

def parse_espn_date(value):
    """Parse ESPN timestamps like '2026-08-29T16:00Z' into timezone-aware datetimes."""
    ts = pd.Timestamp(value)
    if ts.tzinfo is None:
        ts = ts.tz_localize("UTC")
    return ts.to_pydatetime()

def api_get(url, params=None, retries=3):
    """GET JSON from ESPN with simple retries. Raises requests.RequestException if all attempts fail."""
    global api_calls
    last_exc = None
    for attempt in range(retries):
        api_calls += 1
        try:
            response = _session.get(url, params=params, timeout=30)
            response.raise_for_status()
            data = response.json()
            time.sleep(REQUEST_DELAY)
            return data
        except requests.RequestException as exc:
            last_exc = exc
            time.sleep(1.5 * (attempt + 1))
    raise last_exc

## 2. Database

In [2]:
conn = sqlite3.connect(DB_FILE)
conn.execute("PRAGMA foreign_keys = ON")
 
conn.executescript("""
CREATE TABLE IF NOT EXISTS Teams (
    team_id TEXT PRIMARY KEY,
    display_name TEXT NOT NULL,
    short_name TEXT,
    conference TEXT,
    conference_id TEXT
);
CREATE TABLE IF NOT EXISTS Games (
    game_id TEXT PRIMARY KEY,
    game_date TEXT,
    name TEXT,
    week INTEGER,
    home_team_id TEXT REFERENCES Teams(team_id),
    away_team_id TEXT REFERENCES Teams(team_id),
    home_score INTEGER,
    away_score INTEGER,
    status TEXT,
    venue TEXT,
    neutral_site INTEGER
);
CREATE TABLE IF NOT EXISTS Box_Scores (
    game_id TEXT NOT NULL REFERENCES Games(game_id),
    team_id TEXT NOT NULL REFERENCES Teams(team_id),
    stat_name TEXT NOT NULL,
    stat_label TEXT,
    stat_value REAL,
    display_value TEXT,
    PRIMARY KEY (game_id, team_id, stat_name)
);
CREATE TABLE IF NOT EXISTS Player_Box_Scores (
    game_id TEXT NOT NULL REFERENCES Games(game_id),
    team_id TEXT NOT NULL REFERENCES Teams(team_id),
    player_id TEXT NOT NULL,
    player_name TEXT,
    stat_category TEXT NOT NULL,
    stat_name TEXT NOT NULL,
    stat_label TEXT,
    stat_value REAL,
    display_value TEXT,
    PRIMARY KEY (game_id, team_id, player_id, stat_category, stat_name)
);
CREATE INDEX IF NOT EXISTS idx_player_box_scores_team
    ON Player_Box_Scores(team_id);
CREATE INDEX IF NOT EXISTS idx_player_box_scores_category
    ON Player_Box_Scores(stat_category, stat_name);
-- Raw ESPN "boxscore" section per game. payload is zlib-compressed JSON.
-- payload/payload_hash are NULL when ESPN had not published a box score at fetch time.
CREATE TABLE IF NOT EXISTS Raw_Boxscores (
    game_id TEXT PRIMARY KEY REFERENCES Games(game_id),
    fetched_at TEXT NOT NULL,
    payload_hash TEXT,
    payload BLOB
);
CREATE TABLE IF NOT EXISTS Scoreboard_Fetches (
    game_date TEXT PRIMARY KEY,   -- YYYY-MM-DD
    fetched_at TEXT NOT NULL
);
CREATE TABLE IF NOT EXISTS Player_Boxscore_Fetches (
    game_id TEXT PRIMARY KEY REFERENCES Games(game_id),
    payload_hash TEXT
);
-- Raw ESPN "drives" + "scoringPlays" sections per game. payload is zlib-compressed JSON.
-- payload/payload_hash are NULL when ESPN had not published play-by-play at fetch time.
CREATE TABLE IF NOT EXISTS Raw_Plays (
    game_id TEXT PRIMARY KEY REFERENCES Games(game_id),
    fetched_at TEXT NOT NULL,
    payload_hash TEXT,
    payload BLOB
);
-- One row per play, parsed from Raw_Plays (columns match map_schema.flatten_plays).
CREATE TABLE IF NOT EXISTS Plays (
    game_id TEXT NOT NULL REFERENCES Games(game_id),
    drive_number INTEGER NOT NULL,
    drive_id TEXT,
    drive_team_id TEXT,
    drive_team_name TEXT,
    drive_result TEXT,
    drive_description TEXT,
    play_number_in_drive INTEGER NOT NULL,
    play_id TEXT,
    sequence_number TEXT,
    period INTEGER,
    clock TEXT,
    play_type_id TEXT,
    play_type_text TEXT,
    play_type_abbreviation TEXT,
    text TEXT,
    start_down INTEGER,
    start_distance INTEGER,
    start_yard_line INTEGER,
    start_yards_to_endzone INTEGER,
    start_team_id TEXT,
    end_down INTEGER,
    end_distance INTEGER,
    end_yard_line INTEGER,
    end_yards_to_endzone INTEGER,
    end_team_id TEXT,
    stat_yardage INTEGER,
    home_score INTEGER,
    away_score INTEGER,
    scoring_play INTEGER,
    is_penalty INTEGER,
    is_turnover INTEGER,
    wallclock TEXT,
    modified TEXT,
    PRIMARY KEY (game_id, drive_number, play_number_in_drive)
);
CREATE INDEX IF NOT EXISTS idx_plays_play_id ON Plays(game_id, play_id);
-- One row per score, parsed from Raw_Plays. is_offense_score = 1 when the scoring team had
-- the ball at the start of the play (an offensive score), 0 for defensive/return scores
-- and safeties, NULL when the scoring play could not be matched to a play.
CREATE TABLE IF NOT EXISTS Scoring_Plays (
    game_id TEXT NOT NULL REFERENCES Games(game_id),
    score_number INTEGER NOT NULL,
    play_id TEXT,
    period INTEGER,
    clock TEXT,
    team_id TEXT,
    points INTEGER,
    play_type_text TEXT,
    scoring_type TEXT,
    possession_team_id TEXT,
    is_offense_score INTEGER,
    text TEXT,
    PRIMARY KEY (game_id, score_number)
);
-- Derived: team totals of player defensive stats. Fully rebuilt each run.
CREATE TABLE IF NOT EXISTS Team_Defensive_Stats (
    game_id TEXT NOT NULL,
    team_id TEXT NOT NULL,
    stat_category TEXT NOT NULL,
    stat_name TEXT NOT NULL,
    team_total REAL,
    PRIMARY KEY (game_id, team_id, stat_category, stat_name)
);
-- Derived: each team's defense, described by its opponent's team box score. Fully rebuilt each run.
CREATE TABLE IF NOT EXISTS Team_Defense_Opponent_Stats (
    game_id TEXT NOT NULL REFERENCES Games(game_id),
    team_id TEXT NOT NULL REFERENCES Teams(team_id),      -- the defending team
    opponent_id TEXT NOT NULL REFERENCES Teams(team_id),  -- the offense it faced
    stat_name TEXT NOT NULL,
    stat_label TEXT,
    stat_value REAL,
    display_value TEXT,
    PRIMARY KEY (game_id, team_id, stat_name)
);
-- Raw kickoffstatus game pages (compressed). game_id is NULL when the page matched no ESPN game.
CREATE TABLE IF NOT EXISTS Raw_Injury_Pages (
    url TEXT PRIMARY KEY,
    game_id TEXT REFERENCES Games(game_id),
    game_date TEXT NOT NULL,
    lastmod TEXT,
    fetched_at TEXT NOT NULL,
    payload_hash TEXT,
    payload BLOB
);
-- One row per player on a game's availability report, parsed from Raw_Injury_Pages.
-- official = 1: the conference's report; 0: an unofficial note from local media or Covers.
CREATE TABLE IF NOT EXISTS Injury_Reports (
    game_id TEXT NOT NULL REFERENCES Games(game_id),
    team_id TEXT NOT NULL REFERENCES Teams(team_id),
    player_name TEXT NOT NULL,
    official INTEGER NOT NULL,
    jersey TEXT,
    position TEXT,
    status TEXT NOT NULL,
    status_history TEXT,
    report_type TEXT,
    reported_at TEXT,
    source TEXT,
    source_url TEXT,
    note TEXT,
    page_url TEXT NOT NULL REFERENCES Raw_Injury_Pages(url),
    PRIMARY KEY (game_id, team_id, player_name, official)
);
CREATE INDEX IF NOT EXISTS idx_injury_team ON Injury_Reports(team_id, game_id);
CREATE INDEX IF NOT EXISTS idx_box_scores_team ON Box_Scores(team_id);
CREATE INDEX IF NOT EXISTS idx_games_date ON Games(game_date);
""")
conn.commit()
print(f"SQLite database ready: {DB_FILE}")

SQLite database ready: espn_cfb_2026.db


## 3. Teams

Fetched once. FCS opponents are added automatically when they appear in the schedule.

In [3]:
def parse_standings_node(node, parent_conference="", parent_conf_id=""):
    """Recursively walk ESPN's standings tree and return teams with their conference name and ID."""
    teams = []
    current_name = node.get("name", "")
    current_id = str(node.get("id", ""))
 
    # Top-level league labels must not override the real conference name/ID
    is_root_label = current_name.lower() in ["fbs", "ncaa - football bowl subdivision", "football bowl subdivision"]
    if is_root_label:
        conf_label, conf_id = "", ""
    else:
        conf_label = parent_conference if parent_conference else current_name
        conf_id = parent_conf_id if parent_conf_id else current_id
 
    for entry in node.get("standings", {}).get("entries", []):
        team = entry.get("team", {})
        if team:
            teams.append({
                "team_id": team.get("id"),
                "display_name": team.get("displayName"),
                "short_name": team.get("shortDisplayName"),
                "conference": conf_label,
                "conference_id": conf_id,
            })
 
    for child in node.get("children", []):
        teams.extend(parse_standings_node(child, parent_conference=conf_label, parent_conf_id=conf_id))
    return teams

def ensure_team(team_id, display_name):
    """Make sure a team row exists (needed for foreign keys). Never overwrites an existing row."""
    if team_id is not None:
        conn.execute(
            "INSERT OR IGNORE INTO Teams (team_id, display_name) VALUES (?, ?)",
            (str(team_id), display_name or "Unknown"),
        )

def load_fbs_teams(force=False):
    stored = conn.execute(
        "SELECT COUNT(*) FROM Teams WHERE conference IS NOT NULL AND conference != ''"
    ).fetchone()[0]
    if stored and not force:
        print(f"Teams: {stored} FBS teams already stored, no fetch needed.")
        return
 
    data = api_get(STANDINGS_URL, params={"group": "80"})  # 80 = FBS
    teams = {t["team_id"]: t for t in parse_standings_node(data) if t["team_id"]}
    conn.executemany(
        """INSERT INTO Teams (team_id, display_name, short_name, conference, conference_id)
           VALUES (?, ?, ?, ?, ?)
           ON CONFLICT(team_id) DO UPDATE SET
               display_name = excluded.display_name, short_name = excluded.short_name,
               conference = excluded.conference, conference_id = excluded.conference_id""",
        [(str(t["team_id"]), t["display_name"], t["short_name"],
          t["conference"] or None, t["conference_id"] or None) for t in teams.values()],
    )
    conn.commit()
    print(f"Teams: loaded {len(teams)} FBS teams.")

## 4. Schedule (Daily Scoreboard)

In [4]:
GAME_COLUMNS = ["game_id", "game_date", "name", "week", "home_team_id", "away_team_id",
                "home_score", "away_score", "status", "venue", "neutral_site"]
 
GAME_UPSERT = """
INSERT INTO Games (game_id, game_date, name, week, home_team_id, away_team_id,
                   home_score, away_score, status, venue, neutral_site)
VALUES (:game_id, :game_date, :name, :week, :home_team_id, :away_team_id,
        :home_score, :away_score, :status, :venue, :neutral_site)
ON CONFLICT(game_id) DO UPDATE SET
    game_date = excluded.game_date, name = excluded.name, week = excluded.week,
    home_team_id = excluded.home_team_id, away_team_id = excluded.away_team_id,
    home_score = excluded.home_score, away_score = excluded.away_score,
    status = excluded.status, venue = excluded.venue, neutral_site = excluded.neutral_site
"""

def int_or_none(value):
    try:
        return int(value)
    except (TypeError, ValueError):
        return None

def parse_event(event):
    competition = (event.get("competitions") or [{}])[0]
    competitors = competition.get("competitors", [])
    home = next((c for c in competitors if c.get("homeAway") == "home"), {})
    away = next((c for c in competitors if c.get("homeAway") == "away"), {})
    home_team, away_team = home.get("team", {}), away.get("team", {})
    neutral = competition.get("neutralSite")
    return {
        "game_id": str(event.get("id")),
        "game_date": event.get("date"),
        "name": event.get("name"),
        "week": int_or_none((event.get("week") or {}).get("number")),
        "home_team_id": str(home_team["id"]) if home_team.get("id") is not None else None,
        "home_team": home_team.get("displayName"),
        "away_team_id": str(away_team["id"]) if away_team.get("id") is not None else None,
        "away_team": away_team.get("displayName"),
        "home_score": int_or_none(home.get("score")),
        "away_score": int_or_none(away.get("score")),
        "status": ((event.get("status") or {}).get("type") or {}).get("description"),
        "venue": (competition.get("venue") or {}).get("fullName"),
        "neutral_site": None if neutral is None else int(bool(neutral)),
    }

def scoreboard_dates_to_fetch():
    """Days whose scoreboard is missing or may have changed."""
    now = utcnow()
    today = now.date()
    last_day = min(SEASON_END, today + timedelta(days=LOOKAHEAD_DAYS))
    recent_start = today - timedelta(days=RECENT_DAYS)
    stale_before = now - timedelta(hours=REFRESH_AFTER_HOURS)
    fetched = {d: datetime.fromisoformat(t)
               for d, t in conn.execute("SELECT game_date, fetched_at FROM Scoreboard_Fetches")}
 
    dates, day = [], SEASON_START
    while day <= last_day:
        seen = fetched.get(day.isoformat())
        if seen is None:
            dates.append(day)  # never fetched
        else:
            # Late games end by about noon UTC the next day; anything fetched before that may be incomplete
            games_over_by = datetime.combine(day + timedelta(days=1), dtime(12, 0), tzinfo=timezone.utc)
            if seen < stale_before and (day >= recent_start or seen < games_over_by):
                dates.append(day)
        day += timedelta(days=1)
    return dates

def refresh_schedule():
    stats = Counter()
    for day in scoreboard_dates_to_fetch():
        try:
            data = api_get(SCOREBOARD_URL, params={
                "dates": day.strftime("%Y%m%d"), "groups": "80", "limit": "500"})
        except requests.RequestException:
            stats["failed_days"] += 1  # not recorded, so it is retried next run
            continue
 
        for event in data.get("events", []):
            game = parse_event(event)
            ensure_team(game["home_team_id"], game["home_team"])
            ensure_team(game["away_team_id"], game["away_team"])
 
            old = conn.execute(
                "SELECT " + ", ".join(GAME_COLUMNS[1:]) + " FROM Games WHERE game_id = ?",
                (game["game_id"],)).fetchone()
            new = tuple(game[c] for c in GAME_COLUMNS[1:])
            if old is None:
                stats["new_games"] += 1
            elif old != new:
                stats["changed_games"] += 1
            conn.execute(GAME_UPSERT, game)
 
        conn.execute(
            """INSERT INTO Scoreboard_Fetches (game_date, fetched_at) VALUES (?, ?)
               ON CONFLICT(game_date) DO UPDATE SET fetched_at = excluded.fetched_at""",
            (day.isoformat(), iso(utcnow())))
        conn.commit()
        stats["days_fetched"] += 1
 
    print("Schedule:", ", ".join(f"{k}={stats[k]}" for k in
          ["days_fetched", "new_games", "changed_games", "failed_days"]))
    return stats

## 5. Box Scores and Play-by-Play Fetching
ESPN's `boxscore` section is stored raw (compressed) in `Raw_Boxscores`, and parsed into `Box_Scores`. Parsing is a separate step, so a parsing fix only needs `rebuild_box_scores_from_raw()` and no new requests.

The same summary response also carries the play-by-play (`drives` and `scoringPlays`), so `ingest_box_scores` stores that too (section 7). A game is fetched when either part is missing or due for a re-check.

`stat_value` uses ESPN's numeric `value` when it has one, and otherwise falls back to a numeric `displayValue`. Compound values such as `5-12` (made-attempts) or `31:22` stay `NULL` in `stat_value`; the original text is always in `display_value`.

In [5]:
BOXSCORE_INSERT = """
INSERT INTO Box_Scores (game_id, team_id, stat_name, stat_label, stat_value, display_value)
VALUES (?, ?, ?, ?, ?, ?)
ON CONFLICT(game_id, team_id, stat_name) DO UPDATE SET
    stat_label = excluded.stat_label, stat_value = excluded.stat_value,
    display_value = excluded.display_value
"""

RAW_UPSERT = """
INSERT INTO Raw_Boxscores (game_id, fetched_at, payload_hash, payload) VALUES (?, ?, ?, ?)
ON CONFLICT(game_id) DO UPDATE SET
    fetched_at = excluded.fetched_at, payload_hash = excluded.payload_hash, payload = excluded.payload
"""

def pack_boxscore(boxscore):
    """Return (hash, compressed bytes) for a boxscore dict. The hash is of a canonical JSON form."""
    canonical = json.dumps(boxscore, sort_keys=True, separators=(",", ":")).encode("utf-8")
    return hashlib.sha256(canonical).hexdigest(), zlib.compress(canonical, 9)

def unpack_boxscore(blob):
    return json.loads(zlib.decompress(blob))

def numeric_or_none(value):
    if value is None:
        return None
    try:
        return float(value)
    except (TypeError, ValueError):
        return None

def parse_team_stats(game_id, boxscore):
    """Flatten a boxscore dict into Box_Scores rows."""
    rows = []
    for team in boxscore.get("teams", []):
        team_id = team.get("team", {}).get("id")
        if team_id is None:
            continue
        for stat in team.get("statistics", []):
            stat_name = stat.get("name") or stat.get("label")
            if not stat_name:
                continue
            value = numeric_or_none(stat.get("value"))
            if value is None:
                value = numeric_or_none(stat.get("displayValue"))
            rows.append((game_id, str(team_id), str(stat_name), stat.get("label"),
                         value, stat.get("displayValue")))
    return rows

def store_boxscore(game_id, boxscore, payload_hash, blob, fetched_at):
    """Write the raw payload and its parsed rows together (the caller commits)."""
    for team in boxscore.get("teams", []):
        info = team.get("team", {})
        ensure_team(info.get("id"), info.get("displayName"))
    conn.execute("DELETE FROM Box_Scores WHERE game_id = ?", (game_id,))
    conn.executemany(BOXSCORE_INSERT, parse_team_stats(game_id, boxscore))
    conn.execute(RAW_UPSERT, (game_id, fetched_at, payload_hash, blob))

def rebuild_box_scores_from_raw():
    """Re-parse every stored payload into Box_Scores. No network calls."""
    rows = conn.execute("SELECT game_id, payload FROM Raw_Boxscores WHERE payload IS NOT NULL").fetchall()
    for game_id, blob in rows:
        conn.execute("DELETE FROM Box_Scores WHERE game_id = ?", (game_id,))
        conn.executemany(BOXSCORE_INSERT, parse_team_stats(game_id, unpack_boxscore(blob)))
    conn.commit()
    print(f"Rebuilt Box_Scores from {len(rows)} stored payloads.")

In [6]:
def final_games_with_fetch_state():
    cur = conn.execute("""
        SELECT g.game_id, g.game_date, r.fetched_at, r.payload_hash,
               p.fetched_at AS plays_fetched_at, p.payload_hash AS plays_hash
        FROM Games g
        LEFT JOIN Raw_Boxscores r ON r.game_id = g.game_id
        LEFT JOIN Raw_Plays p ON p.game_id = g.game_id
        WHERE g.status = 'Final'""")
    cols = [c[0] for c in cur.description]
    return [dict(zip(cols, row)) for row in cur.fetchall()]

def part_fetch_reason(fetched_at, payload_hash, game_date, now, missing_label):
    """Why one part of the summary (box score or play-by-play) needs a fetch, or None."""
    if fetched_at is None:
        return missing_label
    stale = now - datetime.fromisoformat(fetched_at) >= timedelta(hours=REFRESH_AFTER_HOURS)
    if payload_hash is None:
        return f"{missing_label}_not_published" if stale else None
    if game_date:
        kicked_off = parse_espn_date(game_date)
        if stale and now - kicked_off <= timedelta(days=RECENT_DAYS):
            return "recent_recheck"
    return None

def fetch_reason(game, now):
    """Why this Final game needs a fetch, or None to skip it. One request covers both parts."""
    return (part_fetch_reason(game["fetched_at"], game["payload_hash"], game["game_date"], now, "missing")
            or part_fetch_reason(game["plays_fetched_at"], game["plays_hash"], game["game_date"], now, "missing_plays"))

def ingest_box_scores():
    stats = Counter()
    failed = []
    now = utcnow()
 
    todo = []
    for game in final_games_with_fetch_state():
        reason = fetch_reason(game, now)
        if reason:
            todo.append((game, reason))
        else:
            stats["skipped"] += 1
 
    for game, reason in todo:
        game_id = game["game_id"]
        try:
            summary = api_get(SUMMARY_URL, params={"event": game_id})
        except requests.RequestException as exc:
            failed.append((game_id, str(exc)))
            continue
 
        boxscore = summary.get("boxscore") or {}
        stamp = iso(utcnow())
        try:
            if not boxscore.get("teams"):
                # Nothing published yet. Record the check (keeping any existing payload) so we don't re-ask right away.
                conn.execute(
                    """INSERT INTO Raw_Boxscores (game_id, fetched_at) VALUES (?, ?)
                       ON CONFLICT(game_id) DO UPDATE SET fetched_at = excluded.fetched_at""",
                    (game_id, stamp))
                stats["unchanged" if game["payload_hash"] else "no_boxscore"] += 1
            else:
                payload_hash, blob = pack_boxscore(boxscore)
                if game["payload_hash"] == payload_hash:
                    conn.execute("UPDATE Raw_Boxscores SET fetched_at = ? WHERE game_id = ?", (stamp, game_id))
                    stats["unchanged"] += 1
                else:
                    store_boxscore(game_id, boxscore, payload_hash, blob, stamp)
                    stats["updated" if game["payload_hash"] else "new"] += 1
            # Same response, so play-by-play costs no extra request (section 7).
            stats["plays_" + ingest_plays_from_summary(game_id, summary, game["plays_hash"], stamp)] += 1
            conn.commit()
        except sqlite3.Error as exc:
            conn.rollback()
            failed.append((game_id, f"database error: {exc}"))
 
    print("Box scores:", ", ".join(f"{k}={stats[k]}" for k in
          ["new", "updated", "unchanged", "no_boxscore", "skipped"]) + f", failed={len(failed)}")
    print("Play-by-play:", ", ".join(f"{k}={stats['plays_' + k]}" for k in
          ["new", "updated", "unchanged", "no_plays"]))
    if failed:
        print("Failed (retried automatically next run):", failed)
    stats["failed"] = len(failed)
    return stats

## 6. Player Boxscores

In [7]:
PLAYER_BS_INSERT = """
INSERT INTO Player_Box_Scores
(game_id, team_id, player_id, player_name, stat_category, stat_name,
 stat_label, stat_value, display_value)
VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?)
"""

def numeric_or_none(value):
    """Keep compound ESPN display values as text rather than inventing numbers."""
    if value is None or isinstance(value, bool):
        return None
    try:
        return float(value)
    except (TypeError, ValueError):
        return None

def parse_player_stats(game_id, boxscore):
    """Yield one row per game/team/player/category/statistic."""
    for team_entry in boxscore.get("players", []):
        team_id = (team_entry.get("team") or {}).get("id")
        if team_id is None:
            continue
        for category in team_entry.get("statistics", []):
            category_name = category.get("name")
            if not category_name:
                continue
            keys = category.get("keys") or []
            labels = category.get("labels") or []
            for entry in category.get("athletes", []):
                athlete = entry.get("athlete") or {}
                player_id = athlete.get("id")
                if player_id is None:
                    continue
                for index, raw_value in enumerate(entry.get("stats") or []):
                    if index >= len(keys) or not keys[index]:
                        continue
                    display = None if raw_value is None else str(raw_value)
                    yield (
                        str(game_id), str(team_id), str(player_id),
                        athlete.get("displayName"), str(category_name), str(keys[index]),
                        labels[index] if index < len(labels) else None,
                        numeric_or_none(raw_value), display,
                    )

def rebuild_player_box_scores_from_raw():
    """Completely separated: rebuilds player stats purely from the cached payloads."""
    cached = conn.execute(
        "SELECT game_id, payload FROM Raw_Boxscores WHERE payload IS NOT NULL"
    ).fetchall()
    
    total_rows = 0
    games_with_players = 0
    
    conn.execute("DELETE FROM Player_Box_Scores")
    for game_id, blob in cached:
        boxscore = unpack_boxscore(blob)
        player_rows = list(parse_player_stats(game_id, boxscore))
        if player_rows:
            games_with_players += 1
            conn.executemany(PLAYER_BS_INSERT, player_rows)
            total_rows += len(player_rows)
            
    conn.commit()
    print(f"Rebuilt Player_Box_Scores: {games_with_players} games, {total_rows} rows from {len(cached)} cached payloads.")
    return {"games": games_with_players, "rows": total_rows}

In [8]:
def store_player_boxscore(game_id, boxscore, payload_hash):
    """Write parsed player rows and update the hash tracker (caller commits)."""
    conn.execute("DELETE FROM Player_Box_Scores WHERE game_id = ?", (game_id,))
    rows = list(parse_player_stats(game_id, boxscore))
    if rows:
        conn.executemany(PLAYER_BS_INSERT, rows)
    conn.execute(
        """INSERT INTO Player_Boxscore_Fetches (game_id, payload_hash) VALUES (?, ?)
           ON CONFLICT(game_id) DO UPDATE SET payload_hash = excluded.payload_hash""",
        (game_id, payload_hash),
    )

def ingest_player_box_scores():
    """Incrementally parse only new or updated raw boxscores into Player_Box_Scores."""
    stats = Counter()
    rows = conn.execute("""
        SELECT r.game_id, r.payload_hash, r.payload, p.payload_hash AS parsed_hash
        FROM Raw_Boxscores r
        LEFT JOIN Player_Boxscore_Fetches p ON p.game_id = r.game_id
        WHERE r.payload IS NOT NULL
    """).fetchall()

    for game_id, raw_hash, blob, parsed_hash in rows:
        if raw_hash == parsed_hash:
            stats["skipped"] += 1
            continue
        store_player_boxscore(game_id, unpack_boxscore(blob), raw_hash)
        stats["updated" if parsed_hash else "new"] += 1

    conn.commit()
    print("Player box scores:", ", ".join(f"{k}={stats[k]}" for k in ["new", "updated", "skipped"]))
    return stats

## 7. Play-by-Play
The `drives` and `scoringPlays` sections of each summary are stored raw (compressed) in `Raw_Plays`, then parsed into two tables:

- `Plays`: one row per play, using `collect_drives` and `flatten_plays` from `map_schema.py`, so the notebook and the schema-mapping script share the same column definitions.
- `Scoring_Plays`: one row per score from ESPN's `scoringPlays` list. Points come from the change in the scoring team's score (so a touchdown includes its PAT or two-point try). Each score is matched by play id to the play itself, and `possession_team_id` is that play's `start.team.id`, the team that had the ball when the play began.

`is_offense_score` is 1 when the scoring team had the ball, so pick-sixes, fumble returns, punt/kickoff returns, blocked kicks and safeties are 0. Possession is used instead of ESPN's play type because the type text isn't reliable (for example, a fumble-return touchdown can be labelled `Fumble Recovery (Opponent)` and a passing touchdown `Pass Reception`). On kickoffs and punts, `start.team.id` is the kicking team, so return touchdowns are correctly not charged to the defense.

Per-play `homeScore`/`awayScore` values are sometimes recorded out of order, so points are taken from `scoringPlays` instead.

A parsing change only needs `rebuild_plays_from_raw()`, with no new requests.

In [9]:
# Columns produced by map_schema.flatten_plays, in Plays table order.
PLAY_COLUMNS = [
    "game_id", "drive_number", "drive_id", "drive_team_id", "drive_team_name", "drive_result",
    "drive_description", "play_number_in_drive", "play_id", "sequence_number", "period", "clock",
    "play_type_id", "play_type_text", "play_type_abbreviation", "text",
    "start_down", "start_distance", "start_yard_line", "start_yards_to_endzone", "start_team_id",
    "end_down", "end_distance", "end_yard_line", "end_yards_to_endzone", "end_team_id",
    "stat_yardage", "home_score", "away_score", "scoring_play", "is_penalty", "is_turnover",
    "wallclock", "modified",
]
PLAY_INSERT = (f"INSERT INTO Plays ({', '.join(PLAY_COLUMNS)}) "
               f"VALUES ({', '.join(':' + c for c in PLAY_COLUMNS)})")

SCORING_PLAY_INSERT = """
INSERT INTO Scoring_Plays (game_id, score_number, play_id, period, clock, team_id, points,
                           play_type_text, scoring_type, possession_team_id, is_offense_score, text)
VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?)
"""

RAW_PLAYS_UPSERT = """
INSERT INTO Raw_Plays (game_id, fetched_at, payload_hash, payload) VALUES (?, ?, ?, ?)
ON CONFLICT(game_id) DO UPDATE SET
    fetched_at = excluded.fetched_at, payload_hash = excluded.payload_hash, payload = excluded.payload
"""

def plays_section(summary):
    """The part of a summary response stored in Raw_Plays, or None if ESPN has no plays yet."""
    section = {"drives": summary.get("drives") or {}, "scoringPlays": summary.get("scoringPlays") or []}
    has_plays = any(drive.get("plays") for drive in collect_drives(section))
    return section if has_plays else None

def parse_plays(game_id, section):
    """Flatten the drives into Plays rows (dicts keyed by PLAY_COLUMNS)."""
    rows = flatten_plays(str(game_id), collect_drives(section))
    return [{c: row.get(c) for c in PLAY_COLUMNS} for row in rows]

def parse_scoring_plays(game_id, section, plays, home_team_id):
    """Turn scoringPlays into Scoring_Plays rows, with points and who had the ball."""
    by_play_id = {p["play_id"]: p for p in plays if p["play_id"] is not None}
    last = {"home": 0, "away": 0}
    rows = []
    for number, score in enumerate(section.get("scoringPlays", []), start=1):
        team_id = (score.get("team") or {}).get("id")
        team_id = str(team_id) if team_id is not None else None
        side = "home" if team_id == home_team_id else "away"
        home_score, away_score = int_or_none(score.get("homeScore")), int_or_none(score.get("awayScore"))
        new_total = home_score if side == "home" else away_score
        points = None if new_total is None else new_total - last[side]
        if home_score is not None and away_score is not None:
            last = {"home": home_score, "away": away_score}

        play = by_play_id.get(str(score.get("id")))
        possession = None
        if play is not None:
            possession = play["start_team_id"] or play["drive_team_id"]
            possession = str(possession) if possession is not None else None
        is_offense = None if possession is None or team_id is None else int(possession == team_id)

        rows.append((
            str(game_id), number, str(score.get("id")), (score.get("period") or {}).get("number"),
            (score.get("clock") or {}).get("displayValue"), team_id, points,
            (score.get("type") or {}).get("text"), (score.get("scoringType") or {}).get("name"),
            possession, is_offense, score.get("text"),
        ))
    return rows

def store_plays(game_id, section, payload_hash, blob, fetched_at):
    """Write the raw play-by-play and its parsed rows together (the caller commits)."""
    home_team_id = conn.execute("SELECT home_team_id FROM Games WHERE game_id = ?", (game_id,)).fetchone()[0]
    plays = parse_plays(game_id, section)
    conn.execute("DELETE FROM Plays WHERE game_id = ?", (game_id,))
    conn.execute("DELETE FROM Scoring_Plays WHERE game_id = ?", (game_id,))
    conn.executemany(PLAY_INSERT, plays)
    conn.executemany(SCORING_PLAY_INSERT, parse_scoring_plays(game_id, section, plays, home_team_id))
    if blob is not None:
        conn.execute(RAW_PLAYS_UPSERT, (game_id, fetched_at, payload_hash, blob))

def ingest_plays_from_summary(game_id, summary, stored_hash, fetched_at):
    """Store play-by-play from a summary response already fetched for the box score.
    Returns new / updated / unchanged / no_plays. The caller commits."""
    section = plays_section(summary)
    if section is None:
        # Nothing published yet. Record the check (keeping any existing payload).
        conn.execute(
            """INSERT INTO Raw_Plays (game_id, fetched_at) VALUES (?, ?)
               ON CONFLICT(game_id) DO UPDATE SET fetched_at = excluded.fetched_at""",
            (game_id, fetched_at))
        return "unchanged" if stored_hash else "no_plays"
    payload_hash, blob = pack_boxscore(section)  # same canonical-JSON hashing and compression
    if payload_hash == stored_hash:
        conn.execute("UPDATE Raw_Plays SET fetched_at = ? WHERE game_id = ?", (fetched_at, game_id))
        return "unchanged"
    store_plays(game_id, section, payload_hash, blob, fetched_at)
    return "updated" if stored_hash else "new"

def rebuild_plays_from_raw():
    """Re-parse every stored play-by-play payload into Plays and Scoring_Plays. No network calls."""
    cached = conn.execute("SELECT game_id, payload FROM Raw_Plays WHERE payload IS NOT NULL").fetchall()
    conn.execute("DELETE FROM Plays")
    conn.execute("DELETE FROM Scoring_Plays")
    for game_id, blob in cached:
        store_plays(game_id, unpack_boxscore(blob), None, None, None)
    conn.commit()
    print(f"Rebuilt Plays and Scoring_Plays from {len(cached)} stored payloads.")

## 8. Team Defensive Stats
Sums each player stat in the `defensive` and `interceptions` categories to one total per game and team. This only reads `Player_Box_Scores` (no ESPN requests), so the table is fully rebuilt every run and can never hold stale or duplicate rows.

`team_total` stays `NULL` when every player value for that stat is `NULL` (e.g. compound display values), rather than becoming `0`.

In [10]:
DEFENSIVE_CATEGORIES = ("defensive", "interceptions")

def build_team_defensive_stats():
    """Rebuild Team_Defensive_Stats from the player-level defensive stats."""
    player_stats = pd.read_sql_query(
        f"""
        SELECT game_id, team_id, stat_category, stat_name, stat_value
        FROM Player_Box_Scores
        WHERE stat_category IN ({",".join("?" * len(DEFENSIVE_CATEGORIES))})
        """,
        conn,
        params=DEFENSIVE_CATEGORIES,
    )

    # Sum numeric statistics for each team in each game.
    # min_count=1 preserves NULL when all values are missing.
    team_stats = (
        player_stats.groupby(["game_id", "team_id", "stat_category", "stat_name"], as_index=False)["stat_value"]
        .sum(min_count=1)
        .rename(columns={"stat_value": "team_total"})
    )

    # Replace previously calculated results so reruns don't create duplicate records.
    conn.execute("DELETE FROM Team_Defensive_Stats")
    team_stats.to_sql("Team_Defensive_Stats", conn, if_exists="append", index=False)
    conn.commit()

    print(f"Team defensive stats: player_rows={len(player_stats):,}, team_rows={len(team_stats):,}, "
          f"games={team_stats['game_id'].nunique()}, teams={team_stats['team_id'].nunique()}")
    return {"player_rows": len(player_stats), "team_rows": len(team_stats)}

## 9. Team Defense from Opponent Box Scores
A second, independent view of team defense, kept separate from the player-stat sums in section 8. In every game, the opponent's offensive team stats in `Box_Scores` are exactly what this team's defense allowed, so each opponent row is copied to the defending team, with `opponent_id` recording who that offense was.

The stats are renamed from the defense's point of view (`DEFENSE_STAT_NAMES`), and values are unchanged:

| Opponent stat (`Box_Scores`) | Defensive stat | Meaning |
| --- | --- | --- |
| `totalYards`, `netPassingYards`, `rushingYards` | `totalYardsAllowed`, `passingYardsAllowed`, `rushingYardsAllowed` | Yards given up |
| `yardsPerPass`, `yardsPerRushAttempt` | `yardsPerPassAllowed`, `yardsPerRushAllowed` | Yards per play given up |
| `firstDowns`, `thirdDownEff`, `fourthDownEff` | `firstDownsAllowed`, `thirdDownEffAllowed`, `fourthDownEffAllowed` | Drives the defense let continue |
| `completionAttempts`, `rushingAttempts` | `completionsAttemptsAllowed`, `rushingAttemptsFaced` | Opponent volume |
| `turnovers`, `interceptions`, `fumblesLost` | `takeaways`, `interceptionsMade`, `fumblesRecovered` | Takeaways forced by this defense |
| `possessionTime`, `totalPenaltiesYards` | `opponentPossessionTime`, `opponentPenaltiesYards` | Opponent context, not strictly defense |
| *(from `Games`)* | `pointsAllowed` | Opponent's final score |
| *(from `Scoring_Plays`)* | `offensePointsAllowed` | Points the opponent scored while it had the ball (no defensive or return touchdowns, no safeties) |

A stat ESPN adds later that is not in the mapping is stored as `opponent<StatName>` (e.g. `opponentSacks`) so it can never be mistaken for an offensive stat. Compound values such as `completionsAttemptsAllowed` keep `stat_value = NULL` with the text in `display_value`, as in `Box_Scores`.

`pointsAllowed` is not in ESPN's team box score, so it is added from the opponent's score in `Games` for Final games.

`offensePointsAllowed` is only written for games whose play-by-play is stored, every score could be matched to a play, and the scores add up to the final score in `Games`. Anything else is left out rather than guessed, so rankings fall back to `pointsAllowed` for that game.

In [11]:
# Opponent's Box_Scores stat -> (defensive stat name, label), from the defending team's point of view.
DEFENSE_STAT_NAMES = {
    "totalYards":          ("totalYardsAllowed",          "Total yards allowed"),
    "netPassingYards":     ("passingYardsAllowed",        "Passing yards allowed"),
    "rushingYards":        ("rushingYardsAllowed",        "Rushing yards allowed"),
    "yardsPerPass":        ("yardsPerPassAllowed",        "Yards per pass allowed"),
    "yardsPerRushAttempt": ("yardsPerRushAllowed",        "Yards per rush allowed"),
    "firstDowns":          ("firstDownsAllowed",          "1st downs allowed"),
    "thirdDownEff":        ("thirdDownEffAllowed",        "3rd down efficiency allowed"),
    "fourthDownEff":       ("fourthDownEffAllowed",       "4th down efficiency allowed"),
    "completionAttempts":  ("completionsAttemptsAllowed", "Comp/Att allowed"),
    "rushingAttempts":     ("rushingAttemptsFaced",       "Rushing attempts faced"),
    "turnovers":           ("takeaways",                  "Takeaways"),
    "interceptions":       ("interceptionsMade",          "Interceptions made"),
    "fumblesLost":         ("fumblesRecovered",           "Fumbles recovered"),
    "possessionTime":      ("opponentPossessionTime",     "Opponent possession"),
    "totalPenaltiesYards": ("opponentPenaltiesYards",     "Opponent penalties"),
}
POINTS_ALLOWED = ("pointsAllowed", "Points allowed")
OFFENSE_POINTS_ALLOWED = ("offensePointsAllowed", "Points allowed to offense")

def defense_names_cte():
    """SQL CTE `defense_names(stat_name, defense_name, defense_label)` plus its parameters."""
    rows = ", ".join("(?, ?, ?)" for _ in DEFENSE_STAT_NAMES)
    params = [value for stat, (name, label) in DEFENSE_STAT_NAMES.items() for value in (stat, name, label)]
    return f"WITH defense_names(stat_name, defense_name, defense_label) AS (VALUES {rows})", params

# Unmapped stats fall back to opponent<StatName> / "Opponent <label>".
DEFENSE_NAME_SQL = "COALESCE(n.defense_name, 'opponent' || UPPER(SUBSTR(b.stat_name, 1, 1)) || SUBSTR(b.stat_name, 2))"
DEFENSE_LABEL_SQL = "COALESCE(n.defense_label, 'Opponent ' || b.stat_label)"

OPPONENT_DEFENSE_INSERT = f"""
INSERT INTO Team_Defense_Opponent_Stats
    (game_id, team_id, opponent_id, stat_name, stat_label, stat_value, display_value)
{{cte}}
SELECT b.game_id,
       CASE WHEN b.team_id = g.home_team_id THEN g.away_team_id ELSE g.home_team_id END,
       b.team_id, {DEFENSE_NAME_SQL}, {DEFENSE_LABEL_SQL}, b.stat_value, b.display_value
FROM Box_Scores b
JOIN Games g ON g.game_id = b.game_id
LEFT JOIN defense_names n ON n.stat_name = b.stat_name
WHERE b.team_id IN (g.home_team_id, g.away_team_id)
"""

POINTS_ALLOWED_INSERT = """
INSERT INTO Team_Defense_Opponent_Stats
    (game_id, team_id, opponent_id, stat_name, stat_label, stat_value, display_value)
SELECT game_id, home_team_id, away_team_id, ?, ?, away_score, CAST(away_score AS TEXT)
FROM Games WHERE status = 'Final' AND away_score IS NOT NULL
UNION ALL
SELECT game_id, away_team_id, home_team_id, ?, ?, home_score, CAST(home_score AS TEXT)
FROM Games WHERE status = 'Final' AND home_score IS NOT NULL
"""

# Points the opposing offense scored with the ball, for games whose scoring plays are complete
# (all matched to a play, and adding up to the Games final score).
OFFENSE_POINTS_INSERT = """
INSERT INTO Team_Defense_Opponent_Stats
    (game_id, team_id, opponent_id, stat_name, stat_label, stat_value, display_value)
WITH sides AS (
    SELECT game_id, away_team_id AS defense, home_team_id AS offense, home_score AS offense_score
    FROM Games WHERE status = 'Final' AND home_score IS NOT NULL
    UNION ALL
    SELECT game_id, home_team_id, away_team_id, away_score
    FROM Games WHERE status = 'Final' AND away_score IS NOT NULL
), scored AS (
    SELECT game_id, team_id,
           SUM(points) AS total_points,
           SUM(CASE WHEN is_offense_score = 1 THEN points ELSE 0 END) AS offense_points,
           SUM(is_offense_score IS NULL OR points IS NULL) AS unmatched
    FROM Scoring_Plays GROUP BY game_id, team_id
)
SELECT s.game_id, s.defense, s.offense, ?, ?,
       COALESCE(sc.offense_points, 0), CAST(COALESCE(sc.offense_points, 0) AS TEXT)
FROM sides s
JOIN Raw_Plays r ON r.game_id = s.game_id AND r.payload IS NOT NULL
LEFT JOIN scored sc ON sc.game_id = s.game_id AND sc.team_id = s.offense
WHERE COALESCE(sc.total_points, 0) = s.offense_score AND COALESCE(sc.unmatched, 0) = 0
"""

def build_team_defense_opponent_stats():
    """Rebuild Team_Defense_Opponent_Stats from the opponents' team box scores and final scores."""
    cte, params = defense_names_cte()
    with conn:
        # Full rebuild: corrected or removed box scores cannot leave stale rows behind.
        conn.execute("DELETE FROM Team_Defense_Opponent_Stats")
        box_rows = conn.execute(OPPONENT_DEFENSE_INSERT.format(cte=cte), params).rowcount
        point_rows = conn.execute(POINTS_ALLOWED_INSERT, POINTS_ALLOWED * 2).rowcount
        offense_point_rows = conn.execute(OFFENSE_POINTS_INSERT, OFFENSE_POINTS_ALLOWED).rowcount
    games, teams = conn.execute(
        "SELECT COUNT(DISTINCT game_id), COUNT(DISTINCT team_id) FROM Team_Defense_Opponent_Stats"
    ).fetchone()
    unmapped = conn.execute(
        f"SELECT DISTINCT stat_name FROM Box_Scores WHERE stat_name NOT IN ({','.join('?' * len(DEFENSE_STAT_NAMES))})",
        list(DEFENSE_STAT_NAMES),
    ).fetchall()
    print(f"Team defense from opponents: box_score_rows={box_rows:,}, points_rows={point_rows:,}, "
          f"offense_points_rows={offense_point_rows:,}, games={games}, teams={teams}")
    if unmapped:
        print("  Not in DEFENSE_STAT_NAMES (stored as opponent<StatName>):", [s for (s,) in unmapped])
    return {"box_score_rows": box_rows, "points_rows": point_rows, "offense_points_rows": offense_point_rows}

## 10. Injury (Availability) Reports
ESPN publishes no college football injury data. Its injury endpoints are empty, and game summaries have no injuries section. The conferences do publish official **availability reports**, for conference games only: three days, two days and one day before kickoff, then a final report on gameday. [kickoffstatus.com](https://kickoffstatus.com/) collects these reports for 10 FBS conferences, one page per game, and links each page to the official source. Its robots.txt allows crawling. It is not affiliated with any conference, so spot-check it against the official report now and then.

What each game page gives, and where it goes in `Injury_Reports`:

| On the page | Column |
| --- | --- |
| Player, jersey number, position | `player_name`, `jersey`, `position` |
| Status on the latest report: `Out`, `Doubtful`, `Questionable`, `Probable`, `Game-time decision`, `Available` (listed, then cleared) | `status` |
| Every report's status for that player, e.g. `Init: Quest. → Upd: Quest. → Upd: Out → GD: Out` | `status_history` |
| Latest report type (`Initial`, `Update`, `Game Day`, or `Pending` before the first one) and when it was posted | `report_type`, `reported_at` (UTC) |
| Which conference's report it is, with a link | `source`, `source_url` |
| "Unofficial Injury Notes" from local media or Covers, also for non-conference games | `official = 0`, with the text in `note` and the outlet and date in `source` |

Players not on an official report are available. The site has no 2025 pages, so this only fills the 2026 database.

**When pages are fetched.** The sitemap (one request) lists every game page with a `lastmod`. A page is fetched if we never stored it. It is fetched again if the game is upcoming or within the last `RECENT_DAYS`, and the sitemap says it changed. Older games are never refetched: a report can't change after kickoff, and the site regularly bumps `lastmod` on old pages when it rebuilds.

In [12]:
import html
import re
import unicodedata
from zoneinfo import ZoneInfo

INJURY_SITE = "https://kickoffstatus.com"
INJURY_HEADERS = {"User-Agent": "cfb-ranker-research-notebook (college course project)"}
INJURY_REQUEST_DELAY = 0.5   # seconds between kickoffstatus requests
PACIFIC = ZoneInfo("America/Los_Angeles")   # kickoffstatus shows every time in Pacific time
injury_calls = 0

INJURY_PAGE_UPSERT = """
INSERT INTO Raw_Injury_Pages (url, game_date, lastmod, fetched_at, payload_hash, payload)
VALUES (?, ?, ?, ?, ?, ?)
ON CONFLICT(url) DO UPDATE SET
    game_date = excluded.game_date, lastmod = excluded.lastmod, fetched_at = excluded.fetched_at,
    payload_hash = excluded.payload_hash, payload = excluded.payload
"""

INJURY_INSERT = """
INSERT INTO Injury_Reports (game_id, team_id, player_name, official, jersey, position, status,
                            status_history, report_type, reported_at, source, source_url, note, page_url)
VALUES (:game_id, :team_id, :player_name, :official, :jersey, :position, :status,
        :status_history, :report_type, :reported_at, :source, :source_url, :note, :page_url)
ON CONFLICT(game_id, team_id, player_name, official) DO NOTHING
"""


def injury_get(url):
    global injury_calls
    injury_calls += 1
    response = requests.get(url, headers=INJURY_HEADERS, timeout=30)
    time.sleep(INJURY_REQUEST_DELAY)
    response.raise_for_status()
    return response.text


def injury_sitemap():
    """Every game page kickoffstatus lists: {url: (game date from the URL, lastmod)}."""
    xml = injury_get(f"{INJURY_SITE}/sitemap.xml")
    pages = {}
    for url, lastmod in re.findall(r"<loc>([^<]*/games/[^<]*)</loc>\s*<lastmod>([^<]*)</lastmod>", xml):
        slug_date = re.search(r"/games/(\d{4}-\d{2}-\d{2})-", url)
        if slug_date:
            pages[url] = (slug_date.group(1), lastmod)
    return pages


def html_text(fragment):
    """Inner text of an HTML fragment: tags removed, entities decoded, whitespace collapsed."""
    return re.sub(r"\s+", " ", html.unescape(re.sub(r"<[^>]+>", " ", fragment))).strip()


def pacific_to_utc(stamp, game_date):
    """'Sat Nov 21, 11:00 AM PT' (no year) -> UTC ISO string. The year is the one closest to the game."""
    if not stamp:
        return None
    stamp = re.sub(r"\s*·\s*", ", ", stamp.replace(" PT", "").strip())
    game = date.fromisoformat(game_date)
    for fmt in ("%a %b %d, %I:%M %p", "%a %b %d"):
        try:
            parsed = datetime.strptime(f"{game.year} {stamp}", f"%Y {fmt}")
        except ValueError:
            continue
        candidates = [parsed.replace(year=game.year + d) for d in (-1, 0, 1)]
        best = min(candidates, key=lambda c: abs((c.date() - game).days))
        return best.replace(tzinfo=PACIFIC).astimezone(timezone.utc).strftime("%Y-%m-%dT%H:%MZ")
    return None


def parse_injury_page(page):
    """One kickoffstatus game page -> game info plus one dict per listed player."""
    page = re.sub(r"<script.*?</script>|<style.*?</style>", "", page, flags=re.S)
    h1 = re.search(r"<h1>(.*?)</h1>", page, re.S)
    title = re.match(r"(.+?) (?:at|vs\.?) (.+?) Injury Report", html_text(h1.group(1)) if h1 else "")
    latest = re.search(r"Latest: <b>(.*?)</b>(?: posted ([^<]+))?", page)
    credit = re.search(r'<div class="credit">Source: (.*?) —\s*<a href="([^"]+)"', page, re.S)
    game = {
        "away": title.group(1) if title else None,
        "home": title.group(2) if title else None,
        "report_type": html_text(latest.group(1)) if latest else None,
        "posted": latest.group(2).strip() if latest and latest.group(2) else None,
        "source": html_text(credit.group(1)) if credit else None,
        "source_url": credit.group(2) if credit else None,
    }

    players = []
    official = re.search(r'<div class="teams2">(.*?)(?:<section class="card unofficial"|<h2>Status changes|</main>)',
                         page, re.S)
    for card in re.split(r'<div class="card"><h3>', official.group(1))[1:] if official else []:
        team = html_text(card.split("</a>")[0])
        for row in re.findall(r"<tr><td>(.*?)</tr>", card, re.S):
            cells = re.findall(r"<td[^>]*>(.*?)</td>", "<td>" + row, re.S)
            jersey = re.search(r'<span class="num">#?([^<]*)</span>', cells[0])
            players.append({
                "team": team, "official": 1,
                "player_name": html_text(re.sub(r'<span class="num">.*?</span>', "", cells[0])),
                "jersey": jersey.group(1).strip() if jersey else None,
                "position": html_text(cells[1]) or None,
                "status": html_text(cells[2]),
                "status_history": html_text(cells[3]) if len(cells) > 3 else None,
                "note": None, "note_source": None,
            })
    unofficial = re.search(r'<section class="card unofficial".*?<tbody>(.*?)</tbody>', page, re.S)
    for row in re.findall(r"<tr>(.*?)</tr>", unofficial.group(1), re.S) if unofficial else []:
        cells = re.findall(r"<td[^>]*>(.*?)</td>", row, re.S)
        name, _, team = cells[0].partition('<div class="small muted">')
        players.append({
            "team": html_text(team), "official": 0, "player_name": html_text(name), "jersey": None,
            "position": html_text(cells[1]) or None, "status": html_text(cells[2]).capitalize(),
            "status_history": None, "note": html_text(cells[3]),
            "note_source": html_text(cells[4]) if len(cells) > 4 else None,
        })
    game["players"] = players
    return game

Each page is matched to an ESPN game by date (Pacific time, within a day) and both team names. A name matches an ESPN team when it equals the short or display name after normalizing (case, accents, "State" -> "St"), when it starts the display name ("Tennessee" -> "Tennessee Volunteers"), or through `INJURY_TEAM_ALIASES`. Each player is then assigned to whichever of the two teams his card names. Pages that match no game stay in `Raw_Injury_Pages` with a NULL `game_id` and are listed in the checks.

In [13]:
# kickoffstatus names that don't match an ESPN short or display name after normalizing
INJURY_TEAM_ALIASES = {
    "wku": "98", "mtsu": "2393", "fiu": "2229", "utep": "2638", "utsa": "2636", "uconn": "41",
    "umass": "113", "ul monroe": "2433", "ulm": "2433", "southern miss": "2572", "app st": "2026",
    "app state": "2026", "hawaii": "62", "san jose st": "23", "louisiana lafayette": "309",
    "mississippi val": "2400", "central conn st": "2115",   # FCS abbreviations
}


def norm_team(name):
    name = unicodedata.normalize("NFKD", html.unescape(name or "")).encode("ascii", "ignore").decode()
    name = re.sub(r"[^a-z0-9& ]", "", name.lower().replace("-", " "))
    name = re.sub(r"\bstate\b", "st", name)
    return re.sub(r"\s+", " ", name).strip()


def team_match_score(name, team_id):
    """How well a kickoffstatus team name fits an ESPN team (0 = no match)."""
    n = norm_team(name)
    if INJURY_TEAM_ALIASES.get(n) == team_id:
        return 3
    short, display = INJURY_TEAMS.get(team_id, ("", ""))
    if n in (short, display):
        return 3
    if display.startswith(n + " "):
        return 2
    return 0


def match_injury_game(game_date, away, home):
    """ESPN game_id for a kickoffstatus page, or None. Dates are compared in Pacific time."""
    day = date.fromisoformat(game_date)
    best, best_score = None, 0
    for game_id, espn_day, home_id, away_id in INJURY_GAME_INDEX:
        if abs((espn_day - day).days) > 1:
            continue
        for a, h in ((away_id, home_id), (home_id, away_id)):   # neutral sites can list either order
            sa, sh = team_match_score(away, a), team_match_score(home, h)
            score = sa + sh - abs((espn_day - day).days) - (0.5 if a == home_id else 0)
            if sa and sh and score > best_score:
                best, best_score = game_id, score
    return best


def load_injury_lookups():
    global INJURY_TEAMS, INJURY_GAME_INDEX, INJURY_GAME_DAY
    INJURY_TEAMS = {tid: (norm_team(short), norm_team(display))
                    for tid, display, short in conn.execute("SELECT team_id, display_name, short_name FROM Teams")}
    INJURY_GAME_INDEX = []
    for game_id, game_date, home_id, away_id in conn.execute(
            "SELECT game_id, game_date, home_team_id, away_team_id FROM Games WHERE game_date IS NOT NULL"):
        kickoff = datetime.fromisoformat(game_date.replace("Z", "+00:00")).astimezone(PACIFIC)
        INJURY_GAME_INDEX.append((game_id, kickoff.date(), home_id, away_id))
    INJURY_GAME_DAY = {g[0]: g[1] for g in INJURY_GAME_INDEX}


def store_injury_page(url, game_date, payload):
    """Parse one stored page into Injury_Reports (caller commits). Returns the matched game_id or None."""
    page = parse_injury_page(payload)
    game_id = match_injury_game(game_date, page["away"], page["home"])
    conn.execute("UPDATE Raw_Injury_Pages SET game_id = ? WHERE url = ?", (game_id, url))
    conn.execute("DELETE FROM Injury_Reports WHERE page_url = ?", (url,))
    if game_id is None:
        return None
    # A rescheduled game can have a page for each date. Keep the one whose date matches the kickoff.
    gap = lambda d: abs((date.fromisoformat(d) - INJURY_GAME_DAY[game_id]).days)
    others = conn.execute("SELECT url, game_date FROM Raw_Injury_Pages WHERE game_id = ? AND url != ?",
                          (game_id, url)).fetchall()
    if any(gap(d) < gap(game_date) for _, d in others):
        return game_id
    for other_url, _ in others:
        conn.execute("DELETE FROM Injury_Reports WHERE page_url = ?", (other_url,))
    home_id, away_id = conn.execute("SELECT home_team_id, away_team_id FROM Games WHERE game_id = ?",
                                    (game_id,)).fetchone()
    reported_at = pacific_to_utc(page["posted"], game_date)
    rows = []
    for p in page["players"]:
        scores = {tid: team_match_score(p["team"], tid) for tid in (home_id, away_id)}
        team_id = max(scores, key=scores.get)
        if not scores[team_id] or not p["player_name"]:
            continue
        rows.append({
            "game_id": game_id, "team_id": team_id, "player_name": p["player_name"], "official": p["official"],
            "jersey": p["jersey"], "position": p["position"], "status": p["status"],
            "status_history": p["status_history"],
            "report_type": page["report_type"] if p["official"] else None,
            "reported_at": reported_at if p["official"] else None,
            "source": page["source"] if p["official"] else p["note_source"],
            "source_url": page["source_url"] if p["official"] else None,
            "note": p["note"], "page_url": url,
        })
    conn.executemany(INJURY_INSERT, rows)
    return game_id


def injury_fetch_reason(stored, game_date, lastmod, today):
    if stored is None:
        return "missing"
    stored_lastmod, has_payload = stored
    recent = date.fromisoformat(game_date) >= today - timedelta(days=RECENT_DAYS)
    if recent and (not has_payload or lastmod > (stored_lastmod or "")):
        return "updated"
    return None


def ingest_injury_reports(force=False):
    """Fetch new or updated kickoffstatus game pages, store them raw, and parse them into Injury_Reports."""
    stats = Counter()
    load_injury_lookups()
    pages = injury_sitemap()
    stored = {url: (lastmod, payload is not None) for url, lastmod, payload in
              conn.execute("SELECT url, lastmod, payload_hash FROM Raw_Injury_Pages")}
    today = date.today()
    for url, (game_date, lastmod) in sorted(pages.items(), key=lambda kv: kv[1][0]):
        reason = "forced" if force else injury_fetch_reason(stored.get(url), game_date, lastmod, today)
        if reason is None:
            stats["up_to_date"] += 1
            continue
        try:
            text = injury_get(url)
        except requests.RequestException as e:
            print(f"Failed {url}: {e}")
            stats["failed"] += 1
            continue
        payload = text.encode("utf-8")
        conn.execute(INJURY_PAGE_UPSERT, (url, game_date, lastmod, datetime.now(timezone.utc).isoformat(),
                                          hashlib.sha256(payload).hexdigest(), zlib.compress(payload, 9)))
        stats[reason] += 1
        if store_injury_page(url, game_date, text) is None:
            stats["unmatched"] += 1
        conn.commit()
    return dict(stats)


def rebuild_injury_reports_from_raw():
    """Re-parse every stored page (after a parsing or matching fix). No requests."""
    load_injury_lookups()
    stats = Counter()
    for url, game_date, payload in conn.execute(
            "SELECT url, game_date, payload FROM Raw_Injury_Pages WHERE payload IS NOT NULL").fetchall():
        matched = store_injury_page(url, game_date, zlib.decompress(payload).decode("utf-8"))
        stats["matched" if matched else "unmatched"] += 1
    conn.commit()
    return dict(stats)

## 11. Update the database

Safe to run as often as you like. Prints how many ESPN requests the run made; a run with nothing new should make none.

In [14]:
def update_database(force_teams=False, injuries=True):
    start_calls = api_calls
    load_fbs_teams(force=force_teams)
    schedule_stats = refresh_schedule()
    box_stats = ingest_box_scores()
    player_box_stats = ingest_player_box_scores()
    defensive_stats = build_team_defensive_stats()
    opponent_defense_stats = build_team_defense_opponent_stats()
    start_injury_calls = injury_calls
    injury_stats = ingest_injury_reports() if injuries else {}
    if injuries:
        print(f"Injury reports: {injury_stats}")
    
    calls = api_calls - start_calls
    print(f"ESPN requests this run: {calls}")
    print(f"kickoffstatus requests this run: {injury_calls - start_injury_calls}")
    return {
        "schedule": schedule_stats, 
        "box_scores": box_stats, 
        "player_box_scores": player_box_stats, 
        "team_defensive_stats": defensive_stats,
        "team_defense_opponent_stats": opponent_defense_stats,
        "injury_reports": injury_stats,
        "api_calls": calls
    }
 
update_database()

Teams: 138 FBS teams already stored, no fetch needed.
Schedule: days_fetched=22, new_games=0, changed_games=2, failed_days=0
Box scores: new=2, updated=38, unchanged=27, no_boxscore=0, skipped=757, failed=0
Play-by-play: new=2, updated=5, unchanged=60, no_plays=0
Player box scores: new=2, updated=38, skipped=784
Team defensive stats: player_rows=254,814, team_rows=13,969, games=824, teams=238
Team defense from opponents: box_score_rows=24,720, points_rows=1,648, offense_points_rows=1,646, games=824, teams=238
Injury reports: {'up_to_date': 592, 'updated': 13}
ESPN requests this run: 89
kickoffstatus requests this run: 14


{'schedule': Counter({'days_fetched': 22, 'changed_games': 2}),
 'box_scores': Counter({'skipped': 757,
          'plays_unchanged': 60,
          'updated': 38,
          'unchanged': 27,
          'plays_updated': 5,
          'new': 2,
          'plays_new': 2,
          'failed': 0}),
 'player_box_scores': Counter({'skipped': 784, 'updated': 38, 'new': 2}),
 'team_defensive_stats': {'player_rows': 254814, 'team_rows': 13969},
 'team_defense_opponent_stats': {'box_score_rows': 24720,
  'points_rows': 1648,
  'offense_points_rows': 1646},
 'injury_reports': {'up_to_date': 592, 'updated': 13},
 'api_calls': 89}

## 12. Checks

In [15]:
final_not_stored = conn.execute("""
    SELECT COUNT(*) FROM Games g
    LEFT JOIN Raw_Boxscores r ON r.game_id = g.game_id
    WHERE g.status = 'Final' AND r.payload_hash IS NULL""").fetchone()[0]
print("Final games without a stored box score:", final_not_stored)
 
print("FBS teams with no games:", conn.execute("""
    SELECT t.display_name FROM Teams t
    WHERE t.conference IS NOT NULL AND t.conference != ''
      AND NOT EXISTS (SELECT 1 FROM Games g WHERE t.team_id IN (g.home_team_id, g.away_team_id))
""").fetchall())
 
print("Foreign key violations:", conn.execute("PRAGMA foreign_key_check").fetchall())
 
print("Row counts (Teams, Games, Box_Scores, Raw_Boxscores, Player_Box_Scores, Team_Defensive_Stats, Team_Defense_Opponent_Stats, Raw_Plays, Plays, Scoring_Plays):", conn.execute("""
    SELECT (SELECT COUNT(*) FROM Teams), (SELECT COUNT(*) FROM Games),
           (SELECT COUNT(*) FROM Box_Scores), (SELECT COUNT(*) FROM Raw_Boxscores),
           (SELECT COUNT(*) FROM Player_Box_Scores), (SELECT COUNT(*) FROM Team_Defensive_Stats),
           (SELECT COUNT(*) FROM Team_Defense_Opponent_Stats), (SELECT COUNT(*) FROM Raw_Plays),
           (SELECT COUNT(*) FROM Plays), (SELECT COUNT(*) FROM Scoring_Plays)""").fetchone())

Final games without a stored box score: 0
FBS teams with no games: []
Foreign key violations: []
Row counts (Teams, Games, Box_Scores, Raw_Boxscores, Player_Box_Scores, Team_Defensive_Stats, Team_Defense_Opponent_Stats, Raw_Plays, Plays, Scoring_Plays): (240, 909, 24720, 824, 420791, 13969, 28014, 824, 146661, 7362)


A stat should only have fewer numeric rows than total rows if it is a compound value (for example `completionAttempts`, `totalPenaltiesYards`) or, for `fourthDownEff`, a team with no attempts.

In [16]:
pd.read_sql_query("""
SELECT stat_name, COUNT(*) AS rows_, COUNT(stat_value) AS numeric_rows
FROM Box_Scores
GROUP BY stat_name
ORDER BY stat_name
""", conn)

,stat_name,rows_,numeric_rows
0,completionAttempts,1648,0
1,firstDowns,1648,1648
2,fourthDownEff,1648,1376
3,fumblesLost,1648,1648
4,interceptions,1648,1648
5,netPassingYards,1648,1648
6,possessionTime,1648,1648
7,rushingAttempts,1648,1648
8,rushingYards,1648,1648
9,thirdDownEff,1648,1648


Spot check of team defensive totals: Tennessee Volunteers (team `2633`) in game `401856745` (Kentucky at Tennessee, Nov 7).

In [17]:
pd.read_sql_query("""
SELECT stat_category, stat_name, team_total
FROM Team_Defensive_Stats
WHERE game_id = '401856745' AND team_id = '2633'
ORDER BY stat_category, stat_name
""", conn)

,stat_category,stat_name,team_total
0,defensive,defensiveTouchdowns,0.0
1,defensive,hurries,6.0
2,defensive,passesDefended,3.0
3,defensive,sacks,4.0
4,defensive,soloTackles,42.0
5,defensive,tacklesForLoss,8.0
6,defensive,totalTackles,80.0
7,interceptions,interceptionTouchdowns,0.0
8,interceptions,interceptionYards,0.0
9,interceptions,interceptions,1.0


Each `Box_Scores` row should appear exactly once as an opponent stat (so the first two counts match and nothing is unmatched), and every Final game should give both teams a `pointsAllowed` row.

In [18]:
print("Box_Scores rows vs. opponent-defense rows (excluding points):", conn.execute("""
    SELECT (SELECT COUNT(*) FROM Box_Scores),
           (SELECT COUNT(*) FROM Team_Defense_Opponent_Stats WHERE stat_name NOT IN (?, ?))""",
    (POINTS_ALLOWED[0], OFFENSE_POINTS_ALLOWED[0])).fetchone())

cte, params = defense_names_cte()
print("Box_Scores rows with no matching opponent-defense row:", conn.execute(f"""
    {cte}
    SELECT COUNT(*) FROM Box_Scores b
    LEFT JOIN defense_names n ON n.stat_name = b.stat_name
    WHERE NOT EXISTS (SELECT 1 FROM Team_Defense_Opponent_Stats d
                      WHERE d.game_id = b.game_id AND d.opponent_id = b.team_id
                        AND d.stat_name = {DEFENSE_NAME_SQL}
                        AND d.stat_value IS b.stat_value AND d.display_value IS b.display_value)
""", params).fetchone()[0])

print("Final games vs. points-allowed rows:", conn.execute("""
    SELECT (SELECT 2 * COUNT(*) FROM Games WHERE status = 'Final'),
           (SELECT COUNT(*) FROM Team_Defense_Opponent_Stats WHERE stat_name = ?)""",
    (POINTS_ALLOWED[0],)).fetchone())

Box_Scores rows vs. opponent-defense rows (excluding points): (24720, 24720)
Box_Scores rows with no matching opponent-defense row: 0
Final games vs. points-allowed rows: (1648, 1648)


Spot check of the opponent-based view for the same game: what Tennessee's defense allowed against Kentucky.

In [19]:
pd.read_sql_query("""
SELECT t.display_name AS team, o.display_name AS opponent, d.stat_name, d.stat_label, d.stat_value, d.display_value
FROM Team_Defense_Opponent_Stats d
JOIN Teams t ON t.team_id = d.team_id
JOIN Teams o ON o.team_id = d.opponent_id
WHERE d.game_id = '401856745' AND d.team_id = '2633'
ORDER BY d.stat_name
""", conn)

,team,opponent,stat_name,stat_label,stat_value,display_value
0,Tennessee Volunteers,Kentucky Wildcats,completionsAttemptsAllowed,Comp/Att allowed,NaN,26/34
1,Tennessee Volunteers,Kentucky Wildcats,firstDownsAllowed,1st downs allowed,20.000000,20
2,Tennessee Volunteers,Kentucky Wildcats,fourthDownEffAllowed,4th down efficiency allowed,0.000000,0-2
3,Tennessee Volunteers,Kentucky Wildcats,fumblesRecovered,Fumbles recovered,2.000000,2
4,Tennessee Volunteers,Kentucky Wildcats,interceptionsMade,Interceptions made,1.000000,1
5,Tennessee Volunteers,Kentucky Wildcats,offensePointsAllowed,Points allowed to offense,6.000000,6
6,Tennessee Volunteers,Kentucky Wildcats,opponentPenaltiesYards,Opponent penalties,NaN,7-50
7,Tennessee Volunteers,Kentucky Wildcats,opponentPossessionTime,Opponent possession,2117.000000,35:17
8,Tennessee Volunteers,Kentucky Wildcats,passingYardsAllowed,Passing yards allowed,224.000000,224
9,Tennessee Volunteers,Kentucky Wildcats,pointsAllowed,Points allowed,6.000000,6


Play-by-play checks. Every Final game should have stored plays. For each game the scoring plays should add up to the final score in `Games`; any game that doesn't is listed, and has no `offensePointsAllowed` rows, so rankings use `pointsAllowed` for it. The last table lists every score that wasn't charged to the defense, to spot-check the possession rule.

In [20]:
from IPython.display import Markdown

print("Final games vs. games with stored play-by-play:", conn.execute("""
    SELECT (SELECT COUNT(*) FROM Games WHERE status = 'Final'),
           (SELECT COUNT(*) FROM Raw_Plays r JOIN Games g ON g.game_id = r.game_id
            WHERE g.status = 'Final' AND r.payload IS NOT NULL)""").fetchone())

print("Games with play-by-play vs. games with offensePointsAllowed:", conn.execute("""
    SELECT (SELECT COUNT(*) FROM Raw_Plays WHERE payload IS NOT NULL),
           (SELECT COUNT(DISTINCT game_id) FROM Team_Defense_Opponent_Stats WHERE stat_name = ?)""",
    (OFFENSE_POINTS_ALLOWED[0],)).fetchone())

print("Scoring plays not matched to a play:", conn.execute(
    "SELECT COUNT(*) FROM Scoring_Plays WHERE is_offense_score IS NULL").fetchone()[0])

display(Markdown("**Games whose scoring plays don't add up to the final score**"))
display(pd.read_sql_query("""
WITH totals AS (
    SELECT game_id, team_id, SUM(points) AS scored FROM Scoring_Plays GROUP BY game_id, team_id
)
SELECT g.game_id, g.name, g.home_score, COALESCE(h.scored, 0) AS home_from_plays,
       g.away_score, COALESCE(a.scored, 0) AS away_from_plays
FROM Games g
JOIN Raw_Plays r ON r.game_id = g.game_id AND r.payload IS NOT NULL
LEFT JOIN totals h ON h.game_id = g.game_id AND h.team_id = g.home_team_id
LEFT JOIN totals a ON a.game_id = g.game_id AND a.team_id = g.away_team_id
WHERE g.status = 'Final'
  AND (COALESCE(h.scored, 0) != g.home_score OR COALESCE(a.scored, 0) != g.away_score)
""", conn))

display(Markdown("**Scores not charged to the defense, by ESPN play type**"))
display(pd.read_sql_query("""
SELECT play_type_text, COUNT(*) AS scores, SUM(points) AS points
FROM Scoring_Plays WHERE is_offense_score = 0
GROUP BY play_type_text ORDER BY scores DESC
""", conn))

Final games vs. games with stored play-by-play: (824, 824)
Games with play-by-play vs. games with offensePointsAllowed: (824, 824)
Scoring plays not matched to a play: 0


**Games whose scoring plays don't add up to the final score**

,game_id,name,home_score,home_from_plays,away_score,away_from_plays
0,401862694,The Citadel Bulldogs at Charlotte 49ers,41,34,43,43
1,401856816,Utah Utes at Iowa State Cyclones,17,11,31,31


**Scores not charged to the defense, by ESPN play type**

,play_type_text,scores,points
0,Interception Return Touchdown,117,814
1,Fumble Return Touchdown,41,284
2,Safety,39,78
3,Punt Return Touchdown,35,244
4,Kickoff Return Touchdown,33,229
5,Fumble Recovery (Opponent),14,98
6,Blocked Punt Touchdown,12,84
7,Pass Interception Return,9,63
8,Sack,8,55
9,Fumble Recovery (Own),6,42


Injury report checks. Every stored page should match an ESPN game; any that don't are listed (fix with `INJURY_TEAM_ALIASES`, then `rebuild_injury_reports_from_raw()`). The status counts are a check that the parser saw every status label. The last table shows the current reports for the Tennessee at Vanderbilt game.

In [21]:
print("Injury pages stored vs. matched to an ESPN game:", conn.execute("""
    SELECT COUNT(*), COUNT(game_id) FROM Raw_Injury_Pages WHERE payload IS NOT NULL""").fetchone())
display(Markdown("**Injury pages that matched no ESPN game**"))
display(pd.read_sql_query("SELECT url, game_date FROM Raw_Injury_Pages WHERE game_id IS NULL ORDER BY game_date", conn))

display(Markdown("**Injury report rows by status**"))
display(pd.read_sql_query("""
    SELECT official, status, COUNT(*) AS rows_, COUNT(DISTINCT game_id) AS games
    FROM Injury_Reports GROUP BY official, status ORDER BY official DESC, rows_ DESC""", conn))

display(Markdown("**Latest reports for Tennessee at Vanderbilt**"))
pd.read_sql_query("""
    SELECT t.short_name AS team, r.player_name, r.position, r.status, r.status_history,
           r.report_type, r.reported_at, r.official
    FROM Injury_Reports r
    JOIN Teams t ON t.team_id = r.team_id
    JOIN Games g ON g.game_id = r.game_id
    WHERE g.home_team_id = '238' AND g.away_team_id = '2633' AND g.game_date >= '2026-11-28'
    ORDER BY r.team_id, r.official DESC, r.status, r.player_name""", conn)

Injury pages stored vs. matched to an ESPN game: (605, 604)


**Injury pages that matched no ESPN game**

,url,game_date
0,https://kickoffstatus.com/games/2026-09-26-cen...,2026-09-26


**Injury report rows by status**

,official,status,rows_,games
0,1,Out,8436,563
1,1,Available,1397,301
2,1,Questionable,895,206
3,1,Game-time decision,534,239
4,1,Not listed,112,25
5,1,Probable,111,35
6,1,Doubtful,34,21
7,1,Out (1st half),7,7
8,0,Out,569,266
9,0,Questionable,280,105


**Latest reports for Tennessee at Vanderbilt**

,team,player_name,position,status,status_history,report_type,reported_at,official
0,Vanderbilt,Aaron Bryant,DL,Out,None,Initial,2026-11-26T01:10Z,1
1,Vanderbilt,Blaze Berlowitz,QB,Out,None,Initial,2026-11-26T01:10Z,1
2,Vanderbilt,Cobey Sellers,CB,Out,None,Initial,2026-11-26T01:10Z,1
3,Vanderbilt,Daniel Vollmer Jr.,LB,Out,None,Initial,2026-11-26T01:10Z,1
4,Vanderbilt,Edwin Kolenge,EDG,Out,None,Initial,2026-11-26T01:10Z,1
5,Vanderbilt,Evan Hampton,RB,Out,None,Initial,2026-11-26T01:10Z,1
6,Vanderbilt,Gunner Givens,OL,Out,None,Initial,2026-11-26T01:10Z,1
7,Vanderbilt,Josiah Broxton,LB,Out,None,Initial,2026-11-26T01:10Z,1
8,Vanderbilt,Clinton Azubuike,OL,Questionable,None,Initial,2026-11-26T01:10Z,1
9,Tennessee,Daune Morris,RB,Out,None,Initial,2026-11-26T01:10Z,1


## 13. Quick look

In [22]:
pd.read_sql_query("""
SELECT g.week, g.game_date, h.display_name AS home, g.home_score,
       a.display_name AS away, g.away_score
FROM Games g
JOIN Teams h ON h.team_id = g.home_team_id
JOIN Teams a ON a.team_id = g.away_team_id
WHERE g.status = 'Final'
ORDER BY g.game_date DESC
LIMIT 10
""", conn)

,week,game_date,home,home_score,away,away_score
0,13,2026-11-25T00:00Z,Eastern Michigan Eagles,17,Kent State Golden Flashes,12
1,13,2026-11-25T00:00Z,Western Michigan Broncos,24,Miami (OH) RedHawks,28
2,12,2026-11-22T03:30Z,Nevada Wolf Pack,16,Hawai'i Rainbow Warriors,8
3,12,2026-11-22T03:30Z,Oregon State Beavers,31,Utah State Aggies,24
4,12,2026-11-22T03:30Z,Arizona State Sun Devils,27,Oklahoma State Cowboys,30
5,12,2026-11-22T02:30Z,Boise State Broncos,20,San Diego State Aztecs,22
6,12,2026-11-22T02:00Z,TCU Horned Frogs,17,Utah Utes,38
7,12,2026-11-22T01:00Z,UAB Blazers,17,UTSA Roadrunners,19
8,12,2026-11-22T00:30Z,Fresno State Bulldogs,49,Colorado State Rams,42
9,12,2026-11-22T00:30Z,California Golden Bears,28,Stanford Cardinal,44


Every stat stored in the database (raw and derived), grouped by table and category, with how many rows each has and how many of those are numeric.

In [23]:
from IPython.display import Markdown, display

all_stats = pd.read_sql_query("""
SELECT 'Box_Scores' AS source, NULL AS stat_category, stat_name,
       MAX(stat_label) AS stat_label, COUNT(*) AS rows_, COUNT(stat_value) AS numeric_rows
FROM Box_Scores GROUP BY stat_name
UNION ALL
SELECT 'Player_Box_Scores', stat_category, stat_name,
       MAX(stat_label), COUNT(*), COUNT(stat_value)
FROM Player_Box_Scores GROUP BY stat_category, stat_name
UNION ALL
SELECT 'Team_Defensive_Stats', stat_category, stat_name,
       NULL, COUNT(*), COUNT(team_total)
FROM Team_Defensive_Stats GROUP BY stat_category, stat_name
UNION ALL
SELECT 'Team_Defense_Opponent_Stats', NULL, stat_name,
       MAX(stat_label), COUNT(*), COUNT(stat_value)
FROM Team_Defense_Opponent_Stats GROUP BY stat_name
ORDER BY source, stat_category, stat_name
""", conn)

all_stats["pct_numeric"] = (100 * all_stats["numeric_rows"] / all_stats["rows_"]).round(1).astype(str) + "%"
all_stats = all_stats.rename(columns={
    "stat_category": "Category", "stat_name": "Stat", "stat_label": "ESPN label",
    "rows_": "Rows", "numeric_rows": "Numeric rows", "pct_numeric": "% numeric",
})

with pd.option_context("display.max_rows", None):
    for source, group in all_stats.groupby("source", sort=False):
        table = group.drop(columns="source").dropna(axis="columns", how="all")
        index = ["Category", "Stat"] if "Category" in table else ["Stat"]
        display(Markdown(f"#### `{source}` ({len(table)} stats)"))
        display(table.set_index(index))

#### `Box_Scores` (15 stats)

,ESPN label,Rows,Numeric rows,% numeric
Stat,,,,
completionAttempts,Comp/Att,1648,0,0.0%
firstDowns,1st Downs,1648,1648,100.0%
fourthDownEff,4th down efficiency,1648,1376,83.5%
fumblesLost,Fumbles lost,1648,1648,100.0%
interceptions,Interceptions thrown,1648,1648,100.0%
netPassingYards,Passing,1648,1648,100.0%
possessionTime,Possession,1648,1648,100.0%
rushingAttempts,Rushing Attempts,1648,1648,100.0%
rushingYards,Rushing,1648,1648,100.0%


#### `Player_Box_Scores` (50 stats)

ESPN label   Rows  \
Category      Stat                                                   
defensive     defensiveTouchdowns                        TD  35925   
              hurries                                QB HUR  35925   
              passesDefended                             PD  35925   
              sacks                                   SACKS  35925   
              soloTackles                              SOLO  35925   
              tacklesForLoss                            TFL  35925   
              totalTackles                              TOT  35925   
fumbles       fumbles                                   FUM   2543   
              fumblesLost                              LOST   2543   
              fumblesRecovered                          REC   2543   
interceptions interceptionTouchdowns                     TD   1113   
              interceptionYards                         YDS   1113   
              interceptions                             INT   1113   
kickReturns   kickReturnTouchdowns                       TD   1550   
              kickReturnYards                           YDS   1550   
              kickReturns                                NO   1550   
              longKickReturn                           LONG   1550   
              yardsPerKickReturn                        AVG   1550   
kicking       extraPointsMade/extraPointAttempts         XP   1717   
              fieldGoalPct                              PCT   1717   
              fieldGoalsMade/fieldGoalAttempts           FG   1717   
              longFieldGoalMade                        LONG   1717   
              totalKickingPoints                        PTS   1717   
passing       adjQBR                                    QBR   2674   
              completions/passingAttempts             C/ATT   2677   
              interceptions                             INT   2677   
              passingTouchdowns                          TD   2677   
              passingYards                              YDS   2677   
              yardsPerPassAttempt                       AVG   2677   
puntReturns   longPuntReturn                           LONG   1242   
              puntReturnTouchdowns                       TD   1242   
              puntReturnYards                           YDS   1242   
              puntReturns                                NO   1242   
              yardsPerPuntReturn                        AVG   1242   
punting       grossAvgPuntYards                         AVG   1769   
              longPunt                                 LONG   1769   
              puntYards                                 YDS   1769   
              punts                                      NO   1769   
              puntsInside20                           In 20   1769   
              touchbacks                                 TB   1769   
receiving     longReception                            LONG  12195   
              receivingTouchdowns                        TD  12195   
              receivingYards                            YDS  12195   
              receptions                                REC  12195   
              yardsPerReception                         AVG  12195   
rushing       longRushing                              LONG   9631   
              rushingAttempts                           CAR   9631   
              rushingTouchdowns                          TD   9631   
              rushingYards                              YDS   9631   
              yardsPerRushAttempt                       AVG   9631   

                                                  Numeric rows % numeric  
Category      Stat                                                        
defensive     defensiveTouchdowns                        35925    100.0%  
              hurries                                    35925    100.0%  
              passesDefended                             35925    100.0%  
              sacks              

#### `Team_Defense_Opponent_Stats` (17 stats)

,ESPN label,Rows,Numeric rows,% numeric
Stat,,,,
completionsAttemptsAllowed,Comp/Att allowed,1648,0,0.0%
firstDownsAllowed,1st downs allowed,1648,1648,100.0%
fourthDownEffAllowed,4th down efficiency allowed,1648,1376,83.5%
fumblesRecovered,Fumbles recovered,1648,1648,100.0%
interceptionsMade,Interceptions made,1648,1648,100.0%
offensePointsAllowed,Points allowed to offense,1646,1646,100.0%
opponentPenaltiesYards,Opponent penalties,1648,0,0.0%
opponentPossessionTime,Opponent possession,1648,1648,100.0%
passingYardsAllowed,Passing yards allowed,1648,1648,100.0%


#### `Team_Defensive_Stats` (10 stats)

Rows  Numeric rows % numeric
Category      Stat                                                
defensive     defensiveTouchdowns     1648          1648    100.0%
              hurries                 1648          1648    100.0%
              passesDefended          1648          1648    100.0%
              sacks                   1648          1648    100.0%
              soloTackles             1648          1648    100.0%
              tacklesForLoss          1648          1648    100.0%
              totalTackles            1648          1648    100.0%
interceptions interceptionTouchdowns   811           811    100.0%
              interceptionYards        811           811    100.0%
              interceptions            811           811    100.0%

## Maintenance

- **Parsing change:** edit `parse_team_stats`, then run `rebuild_box_scores_from_raw()`. No requests needed.
- **Player parsing change:** edit `parse_player_stats`, then run `rebuild_player_box_scores_from_raw()` and `build_team_defensive_stats()`.
- **Play-by-play parsing change:** edit `parse_plays` / `parse_scoring_plays` (or `flatten_plays` in `map_schema.py`), then run `rebuild_plays_from_raw()` and `build_team_defense_opponent_stats()`. No requests needed.
- **Defensive aggregation change:** edit `build_team_defensive_stats` (or `DEFENSIVE_CATEGORIES`) and rerun it.
- **Opponent-based defense change:** edit `build_team_defense_opponent_stats` and rerun it; it only reads `Box_Scores`, `Games`, `Scoring_Plays` and `Raw_Plays`.
- **Injury report parsing or team-matching change:** edit `parse_injury_page`, `INJURY_TEAM_ALIASES` or `match_injury_game`, then run `rebuild_injury_reports_from_raw()`. No requests needed. `ingest_injury_reports(force=True)` refetches every page (about 5 minutes).
- **Force a team refresh:** `update_database(force_teams=True)`.
- **Remove the old derived table** from earlier experiments (derived stats are now computed on demand):

In [24]:
# conn.execute("DROP TABLE IF EXISTS Defense_Stats"); conn.commit()
# rebuild_box_scores_from_raw()
# rebuild_player_box_scores_from_raw()
# rebuild_plays_from_raw()
# build_team_defensive_stats()
# build_team_defense_opponent_stats()
# rebuild_injury_reports_from_raw()